# Exploring FHIR R4 Bundles with pandas

This notebook loads FHIR Bundles from S3 (or a local folder), turns them into one pandas DataFrame per resource type, and shows example queries.

**Using a notebook:** run cells one at a time with **Shift+Enter**, or use *Run → Run All Cells*. Cells share state, so run them in order from the top. If things get confusing, use *Kernel → Restart Kernel and Run All Cells*.

It runs on the synthetic sample data in `data/sample/` by default. To use your own data, change `SOURCE` in the **Parameters** cell below.

In [ ]:
# Make sure the cup_fhir package (in ../src) and its dependencies are available.
# On a fresh clone (e.g. Seqera Studios, JupyterHub) this installs them into the notebook's
# own Python the first time it runs. It does nothing once cup_fhir is installed.
import importlib.util, sys
from pathlib import Path

if importlib.util.find_spec("cup_fhir") is None:
    repo_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
    %pip install --quiet -e "{repo_root}"
    # A new install is only picked up automatically after a kernel restart, so add it for this session too
    sys.path.insert(0, str(repo_root / "src"))

In [ ]:
# Reload cup_fhir automatically when its source files change (no kernel restart needed)
%load_ext autoreload
%autoreload 2

import pandas as pd

from cup_fhir import (
    LOINC, SNOMED,
    load_bundles, validate_bundles, to_dataframes, populated,
    codes, has_code, has_category, identifier_value,
    patient_names_and_ages, with_patient, latest_per_patient,
)

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 80)

## Parameters

In [ ]:
# Where to load from: an s3:// URI or a local folder (relative to this notebook)
SOURCE = "../data/sample"
# SOURCE = "s3://my-bucket/path/to/fhir/"

AWS_PROFILE = None      # e.g. "umccr-dev"; None uses your default AWS credentials

## 1. Load the bundles

`load_bundles` scans `SOURCE` for `.json`, `.ndjson` and gzipped files and keeps everything whose `resourceType` is `Bundle`. Problem files are reported rather than stopping the load.

In [ ]:
result = load_bundles(SOURCE, profile=AWS_PROFILE)
result

In [ ]:
# Files that couldn't be read, and files that contained no Bundles
display(result.errors)
display(result.files_without_bundles)

### Optional: validate against the FHIR R4 models

This parses each Bundle into a typed `fhir.resources` object. Anything listed in `invalid` is still usable as raw JSON through `result.bundles`.

In [ ]:
valid, invalid = validate_bundles(result.bundles)
print(f"{len(valid)} valid, {len(invalid)} invalid")
for key, err in invalid.items():
    print(key, "\n", err, "\n")

In [ ]:
# Typed objects give attribute access and tab-completion
bundle = next(iter(valid.values()))
bundle.type, len(bundle.entry), [e.resource.get_resource_type() for e in bundle.entry][:5]

## 2. One DataFrame per resource type

Nested objects become dotted columns (`subject.reference`, `valueQuantity.value`). Arrays such as `code.coding`, `name` and `identifier` stay as list columns, and the `cup_fhir` helpers query those.

Each row also gets `_source` (which Bundle it came from) and `_fullUrl` (needed to resolve references).

In [ ]:
dfs = to_dataframes(result.bundles)
{resource_type: len(df) for resource_type, df in dfs.items()}

In [ ]:
patients     = patient_names_and_ages(dfs["Patient"])   # adds family, given, age
observations = dfs.get("Observation", pd.DataFrame())
conditions   = dfs.get("Condition", pd.DataFrame())
medreqs      = dfs.get("MedicationRequest", pd.DataFrame())

In [ ]:
# Which fields does this dataset actually fill in? (fraction of rows with a value)
populated(observations)

## 3. Patients

In [ ]:
patients[["id", "family", "given", "gender", "birthDate", "age"]]

In [ ]:
# Women aged 65 and over
patients.query("gender == 'female' and age >= 65")[["id", "family", "given", "age"]]

In [ ]:
# Born in a date range
patients[patients["birthDate"].between("1950-01-01", "1959-12-31")][["id", "family", "birthDate"]]

In [ ]:
# Pull out an identifier by its system URI (change this to your MRN / Medicare / IHI system)
MRN_SYSTEM = "http://example.org/fhir/mrn"
patients["mrn"] = identifier_value(patients["identifier"], MRN_SYSTEM)
patients[patients["mrn"] == "MRN001"][["id", "mrn", "family"]]

## 4. Observations

Coded fields are `CodeableConcept`s: a list of codings, each with a `system` (e.g. LOINC) and a `code`. `has_code` gives a True/False mask to filter on.

In [ ]:
# Parse dates once. utc=True copes with mixed timezone offsets.
observations["effective"] = pd.to_datetime(observations["effectiveDateTime"], format="ISO8601", utc=True)

# What kinds of observation are there?
observations["code.text"].value_counts()

In [ ]:
# All HbA1c results (LOINC 4548-4)
hba1c = observations[has_code(observations["code.coding"], "4548-4", LOINC)]
hba1c[["subject.reference", "effective", "valueQuantity.value", "valueQuantity.unit"]]

In [ ]:
# HbA1c above 8%
hba1c[hba1c["valueQuantity.value"] > 8][["subject.reference", "effective", "valueQuantity.value"]]

In [ ]:
# Any of several codes, e.g. HbA1c or body weight
observations[has_code(observations["code.coding"], {"4548-4", "29463-7"}, LOINC)][["code.text", "valueQuantity.value"]]

In [ ]:
# Final laboratory results within a date window
observations[
    (observations["status"] == "final")
    & has_category(observations["category"], "laboratory")
    & observations["effective"].between("2024-01-01", "2025-01-01", inclusive="left")
][["code.text", "effective", "valueQuantity.value"]]

In [ ]:
# Every code used (shows what you can filter on)
codes(observations["code.coding"]).explode().value_counts()

## 5. Joining resources to their patient

`with_patient` follows `subject.reference` to the Patient and adds `patient_*` columns. It handles both `Patient/123` and `urn:uuid:...` references.

In [ ]:
hba1c_with_patient = with_patient(hba1c, patients)
hba1c_with_patient[["patient_family", "patient_given", "patient_age", "effective", "valueQuantity.value"]]

In [ ]:
# Latest HbA1c per patient
latest_per_patient(hba1c_with_patient, "effective")[["patient_id", "patient_family", "effective", "valueQuantity.value"]]

In [ ]:
# Number of observations per patient
with_patient(observations, patients).groupby(["patient_id", "patient_family"]).size().sort_values(ascending=False)

## 6. Conditions and simple cohorts

In [ ]:
conditions["code.text"].value_counts()

In [ ]:
# Active type 2 diabetes (SNOMED 44054006)
active = has_code(conditions["clinicalStatus.coding"], "active")
t2dm = with_patient(conditions[active & has_code(conditions["code.coding"], "44054006", SNOMED)], patients)
t2dm[["patient_family", "patient_age", "code.text", "onsetDateTime"]]

In [ ]:
# Cohort: active T2DM AND any HbA1c > 8
high_a1c_ids = set(hba1c_with_patient.loc[hba1c_with_patient["valueQuantity.value"] > 8, "patient_id"])
cohort = patients[patients["id"].isin(set(t2dm["patient_id"]) & high_a1c_ids)]
cohort[["id", "family", "given", "age"]]

## 7. Medications

In [ ]:
active_meds = with_patient(medreqs[medreqs["status"] == "active"], patients)
active_meds[["patient_family", "medicationCodeableConcept.text", "authoredOn"]]

In [ ]:
# Free-text match, handy when medication coding is inconsistent
active_meds[active_meds["medicationCodeableConcept.text"].str.contains("metformin", case=False, na=False)]

## Going further

- **Raw JSON** for any Bundle: `result.bundles[key]`, a plain dict.
- **Export** a result for other tools: `cohort.to_csv("cohort.csv")`. Write anything containing real patient data under `data/` (it's git-ignored), never into the repo.
- **Before committing this notebook**, make sure outputs are stripped. The README explains how to set up `nbstripout` so it happens automatically.